In [11]:
# Blocked reaction is a reaction that cannot carry any flux under the current model constraints
import cobra
print(cobra.__version__)
import pandas as pd

0.30.0


In [3]:
model = cobra.io.read_sbml_model("Helicobacter_pylori.xml")

'' is not a valid SBML 'SId'.


In [5]:
from cobra.flux_analysis import find_blocked_reactions

blocked_rxns = find_blocked_reactions(model)

print("Total reactions:", len(model.reactions))
print("Number of blocked reactions:", len(blocked_rxns))

blocked_percentage = (len(blocked_rxns) / len(model.reactions)) * 100

print("Blocked reaction percentage:", round(blocked_percentage, 2), "%")

Total reactions: 1025
Number of blocked reactions: 425
Blocked reaction percentage: 41.46 %


In [13]:
blocked_df = pd.DataFrame({
    "Reaction": blocked_rxns
})

blocked_df.head(20)

,Reaction
0,23DHMPO
1,23PDE2
2,23PDE4
3,23PDE7
4,23PDE9
5,2AHBUTI
6,2HMCOXT
7,2MBCOATA
8,3HAD10M11
9,3HAD10M12


In [15]:
blocked_df.to_csv("H.pylori_Blocked_Reactions.csv", index=False)

print("Saved as H.pylori_Blocked_Reactions.csv")

Saved as H.pylori_Blocked_Reactions.csv


In [17]:
blocked_df = pd.DataFrame({"Reaction": blocked_rxns})

blocked_df["Type"] = blocked_df["Reaction"].apply(
    lambda x:
        "Exchange" if x.startswith("EX_") else
        "Demand" if x.startswith("DM_") else
        "Sink" if x.startswith("SK_") else
        "Biomass" if "biomass" in x.lower() else
        "Internal"
)

print(blocked_df["Type"].value_counts())

Type
Internal    410
Exchange     15
Name: count, dtype: int64


In [19]:
internal_blocked = blocked_df[
    blocked_df["Type"] == "Internal"
]

print("Internal blocked reactions:",
      len(internal_blocked))

print("Internal blocked percentage:",
      round(len(internal_blocked) / len(model.reactions) * 100, 2),
      "%")

Internal blocked reactions: 410
Internal blocked percentage: 40.0 %


In [21]:
internal_ids = internal_blocked["Reaction"].tolist()

internal_blocked_info = []

for rxn_id in internal_ids:
    rxn = model.reactions.get_by_id(rxn_id)

    internal_blocked_info.append({
        "Reaction": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.reaction,
        "Subsystem": rxn.subsystem
    })

internal_blocked_df = pd.DataFrame(internal_blocked_info)

internal_blocked_df.head(20)

,Reaction,Name,Equation,Subsystem
0,23DHMPO,"(R)-2,3-Dihydroxy-3-methylpentanoate:NADP+ oxi...",23dhmp[c] + nadp[c] <=> 3h3mop[c] + h[c] + nad...,
1,23PDE2,"2,3-Cyclic UMP 3-nucleotidohydrolase",23cump[c] + h2o[c] --> 3ump[c] + h[c],
2,23PDE4,"2,3-Cyclic CMP 3-nucleotidohydrolase",23ccmp[c] + h2o[c] --> 3cmp[c] + h[c],
3,23PDE7,"2,3-Cyclic AMP 3-nucleotidohydrolase",23camp[c] + h2o[c] --> 3amp[c] + h[c],
4,23PDE9,"2,3-Cyclic GMP 3-nucleotidohydrolase",23cgmp[c] + h2o[c] --> 3gmp[c] + h[c],
5,2AHBUTI,(S)-2-Aceto-2-hydroxybutanoate isomerase,2ahbut[c] <=> 3h3mop[c],
6,2HMCOXT,4-oxalocrotonate tautomerase,2hmc[c] <=> oxalc[c],
7,2MBCOATA,2-methylbutanoyl-CoA[acyl-carrier-protein] tra...,2mbcoa[c] + ACP[c] <=> 2mbutACP[c] + coa[c],
8,3HAD10M11,10-methyl-3-hydroxy-undecanoyl-ACP hydro-lyase,10m3hundecACP[c] --> 10mtundec2eACP[c] + h2o[c],
9,3HAD10M12,10-methyl-3-hydroxy-dodecanoyl-ACP hydro-lyase,10m3hddcaACP[c] --> 10mtddec2eACP[c] + h2o[c],


In [23]:
subsystem_counts = (
    internal_blocked_df["Subsystem"]
    .fillna("Unknown")
    .value_counts()
)

print(subsystem_counts)

Subsystem
    410
Name: count, dtype: int64
